# 5. Extraindo *Features* de LLMs

In [2]:
# Importando as bibliotecas para lidar com LLMs
import pandas as pd
import ollama
import re

In [14]:
# Caminho do CSV original
input_path = "../data/final/enem_data_embeddings.pkl"
output_path = "../data/final/enem_dataset.pkl"

In [4]:
# Carregar CSV
df = pd.read_pickle(input_path)
df.head()

,numero_questao,enunciado,alternativas,gabarito,ano,gabarito_texto,distratores,enunciado_tokens,gabarito_tokens,distratores_tokens,dificuldade,enunciado_embbedings_word2vec,gabarito_embbedings_word2vec,distratores_embbedings_word2vec
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,C,2009,"reduzir o desmatamento, mantendo-se, assim, o ...",reduzir o calor irradiado pela Terra mediante ...,atmosfera terrestre composta gases nitrogênio ...,reduzir desmatamento mantendo assim potencial ...,reduzir calor irradiado terra mediante substit...,-1.70677,"[-0.0016308315, -0.057879616, -0.085349284, 0....","[0.007537251, -0.049133625, 0.11472875, -0.115...","[0.03476311, -0.035221867, -0.00517779, -0.128..."
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,D,2009,Estimativa de tempo necessário para metaboliza...,Concentração média de álcool no sangue ao long...,analise figura supondo necessário dar título f...,estimativa tempo necessário metabolizar difere...,concentração média álcool sangue longo dia var...,0.62043,"[-0.026465332, -0.030027837, 0.028988913, 0.14...","[0.07856357, -0.022353431, -0.12115115, 0.0183...","[0.0763594, -0.11076818, 0.0048947046, 0.04739..."
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",A,2009,"induzir a imunidade, para proteger o organismo...",ser capaz de alterar o genoma do organismo por...,estima atualmente mundo milhões pessoas infect...,induzir imunidade proteger organismo contamina...,capaz alterar genoma organismo portador induzi...,2.07704,"[0.027941352, 0.008942129, 0.08566157, 0.02149...","[0.097563, -0.0034619968, 0.08206, -0.109324, ...","[0.03503356, -0.010700853, 0.008868624, -0.027..."
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,B,2009,os genótipos idênticos e os fenótipos diferentes.,os genótipos e os fenótipos idênticos.; difere...,experimento preparou conjunto plantas técnica ...,genótipos idênticos fenótipos diferentes,genótipos fenótipos idênticos diferenças genót...,0.11500,"[-0.027700324, -0.014135911, -0.03359886, -0.0...","[0.098122, 0.0499915, -0.214217, -0.0571225, -...","[0.0143338675, -0.014280667, -0.21914314, 0.08..."
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,E,2009,"Kepler apresentou uma teoria científica que, g...","Ptolomeu apresentou as ideias mais valiosas, p...",linha tradição antiga astrônomo grego ptolomeu...,kepler apresentou teoria científica graças mét...,ptolomeu apresentou ideias valiosas serem anti...,0.21694,"[-0.027011229, 0.023866067, -0.08451317, 0.043...","[-0.031293802, 0.018204402, -0.0655475, 0.0612...","[-0.0040200884, 0.03563236, -0.038875517, 0.06..."


---
## 5.1. Configurando modelo LLM

In [5]:
# LLM Utilizado: LLama 3.2 (localmente)
MODEL_NAME = "llama3.2:latest"

# Definindo prompt para o modelo
prompt = """Responda à seguinte questão:

    Enunciado:
    {enunciado}

    Alternativas:
    {alternativas}

    Qual a alternativa correta? Responda apenas com a letra (A, B, C, D ou E)."""

In [6]:
# Função para perguntar ao modelo
def ask_llm(enunciado, alternativas):
    # Construindo mensagem
    message = [
        {
            "role": "user",
            "content": prompt.format(enunciado=enunciado, alternativas=alternativas),
        }
    ]
    try:
        # Enviando prompt para o modelo
        result = ollama.chat(model=MODEL_NAME, messages=message)
        resposta = result["message"]["content"].strip().upper()

        # Capturando a letra da resposta
        match = re.search(r"\b[A-E]\b", resposta.upper())
        resposta_letra = match.group(0) if match else "x"

        return resposta_letra

    except Exception as e:
        print(f"Erro ao consultar modelo: {e}")
        return "?"

---
## 5.2. Aplicando Perguntas ao LLM

In [7]:
df["resposta_llamma"] = df[["enunciado", "alternativas"]].apply(
    lambda row: ask_llm(row["enunciado"], row["alternativas"]), axis=1
)

In [8]:
df["acerto_llamma"] = df[["gabarito", "resposta_llamma"]].apply(
    lambda row: row["gabarito"] == row["resposta_llamma"], axis=1
)

---
## 5.3. Armazenando Resultado

In [15]:
# Salvar resultado
df.to_pickle(output_path)
# pd.(df, output_path, index=False)

In [17]:
df["acerto_llamma"].sum()

171

---